In [0]:
from pyspark.sql.functions import col, round, when, lit

# 1. Read cleaned data from Silver Managed Tables
df_pop = spark.read.table("silver_ibge_population_cleansed")
df_crime = spark.read.table("silver_sinesp_crime_data_cleansed")

# 2. Perform the JOIN between Crimes and Demographics using the clean municipality name and state
# Note: We join on municipality_name_clean to overcome differences in accents or spacing.
df_joined = df_crime.join(
    df_pop,
    df_crime["municipality_name_clean"] == df_pop["municipality_name_clean"],
    "inner"
)

# 3. Calculate the Crime Rate per 100,000 inhabitants
# Formula: (Occurrences / Estimated Population) * 100,000
df_gold_model = df_joined.withColumn(
    "crime_rate_per_100k",
    when(
        (col("population_estimated").isNotNull()) & (col("population_estimated") > 0),
        round((col("occurrences_count") / col("population_estimated")) * 100000, 2)
    ).otherwise(lit(0.0))
)

# 4. Select and organize the final analytical schema for Business Consumption
df_gold_final = df_gold_model.select(
    df_crime["state_code"],
    df_crime["municipality_name_original"].alias("municipality_name"),
    df_crime["crime_type"],
    df_crime["reference_date"],
    df_crime["occurrences_count"],
    df_pop["population_estimated"],
    col("crime_rate_per_100k")
)

# 5. Save to the Gold Layer as a Managed Delta Table
table_name_gold = "gold_municipality_crime_rate"

df_gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name_gold)

print(f"Gold model successfully created! {df_gold_final.count()} records processed and materialized.")
display(df_gold_final.orderBy(col("crime_rate_per_100k").desc()))